<a href="https://colab.research.google.com/github/vintage-alt/fine-tuning/blob/main/Fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from google.colab import userdata
import os

# Authenticate with Hugging Face using the Colab Secret
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

max_seq_length = 2048

# 1. Load Model (Use 1.5B or 7B, Colab T4 can handle 7B in 4-bit)
print("Loading Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 2. Add LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

# 3. Load Dataset STREAMING (No Drive needed!)
print("Streaming dataset from Hugging Face...")
dataset = load_dataset("Vyber07/cyber-security", split="train", streaming=True)
# Take only 5000 rows for a quick training run
dataset = dataset.take(5000)

# 4. Format Prompts
def format_prompt(example):
    instruction = example.get("Instruction", example.get("instruction", "Analyze this."))
    output = example.get("Output", example.get("output", ""))
    if not output and "conversations" in example:
        convs = example["conversations"]
        instruction = next((c.get("value", "") for c in convs if c.get("from") == "human"), "")
        output = next((c.get("value", "") for c in convs if c.get("from") == "gpt"), "")
    if not instruction or not output:
        return {"text": ""}
    text = f"<|im_start|>system\nYou are an expert cybersecurity analyst.<|im_end|>\n<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
    return {"text": text}

print("Formatting dataset...")
dataset = dataset.map(format_prompt)
# Filter out empty ones
dataset = dataset.filter(lambda x: len(x["text"]) > 0)

# 5. Train
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
    ),
)

print("Starting training...")
trainer.train()

# 6. Save
adapter_name = "qwen2.5-7b-cyber-lora-v1-sft5k"
model.save_pretrained(adapter_name)
tokenizer.save_pretrained(adapter_name)
print("Done!")